In [1]:
import pandas as pd
import numpy as np

Using the Path library to read the file

In [2]:
from pathlib import Path

project_root = Path.cwd().parent.parent
data_path = project_root / "data" / "raw" / "billings.csv"

df_bill = pd.read_csv(data_path, header=0, low_memory=False)
df_bill.head(10)

,Co_Ref,Renewal_Month,Connection_Net,Connection_Qty,Discount_Amount,Sustainability_Score,Total_Renewal_Score_New,Starting_Connection_Net,Starting_Connection_Qty,Last_Years_Price,...,Connection_Group,Tenure_Group,#_of_Connection,Last_Renewal,Last_Band,Last_Total_Net_Paid,Last_Connections,Anchor_Group,Renewal_Year,DateTime_Out
0,VT6174,01-11-2024,NaN,NaN,NaN,8.0,42.5,NaN,NaN,799.0,...,1,3,1.0,01-11-2023,Band B,664.0,1.0,1,2024,01-11-2024
1,VD3828,01-08-2025,NaN,NaN,NaN,8.0,41.5,NaN,NaN,799.0,...,1,1,1.0,NaN,NaN,NaN,NaN,1,2025,01-08-2025
2,DV8120,01-03-2025,NaN,NaN,NaN,8.0,33.0,NaN,NaN,799.0,...,1,4+,1.0,01-03-2024,Band C1,749.0,1.0,1,2025,01-03-2025
3,EZ9894,01-06-2025,NaN,NaN,NaN,9.5,44.5,NaN,NaN,799.0,...,1,4+,1.0,01-06-2024,Band C1,749.0,1.0,1,2025,01-06-2025
4,FA8957,01-03-2025,NaN,NaN,NaN,9.5,42.5,NaN,NaN,799.0,...,1,3,1.0,01-03-2024,Band C1,749.0,1.0,1,2025,01-03-2025
5,QS2598,01-06-2025,NaN,NaN,NaN,8.0,40.5,NaN,NaN,799.0,...,1,4+,1.0,01-06-2024,Band C1,749.0,1.0,1,2025,01-06-2025
6,CJ9355,01-03-2025,NaN,NaN,NaN,8.0,43.0,NaN,NaN,799.0,...,1,4+,1.0,01-03-2024,Band C1,749.0,1.0,1,2025,01-03-2025
7,FP1608,01-11-2024,NaN,NaN,NaN,9.5,44.5,NaN,NaN,799.0,...,1,4+,1.0,01-11-2023,Band C1,749.0,1.0,1,2024,01-11-2024
8,TZ9717,01-11-2024,NaN,NaN,NaN,8.0,41.0,NaN,NaN,799.0,...,1,4+,1.0,01-11-2023,Band C1,749.0,1.0,1,2024,01-11-2024
9,UP7099,01-04-2025,NaN,NaN,NaN,8.0,42.0,NaN,NaN,799.0,...,1,3,1.0,01-04-2024,Band C1,749.0,3.0,1,2025,01-04-2025


This dataset contains billing and subscription details per customer prospect. It includes pricing columns, renewal scores,
membership status, audit status, payment methods and prior-year comparisons.
The target column for churn modelling is Prospect_Outcome (Won / Churned / Open).

In [3]:
print(f"Rows: {df_bill.shape[0]}, Columns: {df_bill.shape[1]}")
df_bill.dtypes

Rows: 122082, Columns: 59


Co_Ref                            str
Renewal_Month                     str
Connection_Net                float64
Connection_Qty                float64
Discount_Amount                   str
Sustainability_Score          float64
Total_Renewal_Score_New       float64
Starting_Connection_Net       float64
Starting_Connection_Qty       float64
Last_Years_Price              float64
Last_Years_Date_Paid          float64
Auto_Renewal_Score              int64
Status_Scores                   int64
Anchoring_Score               float64
Tenure_Scores                 float64
Proforma_Auto_Renewal          object
Proforma_World_Pay_Token       object
Proforma_Date                     str
Current_Anchorings              int64
Current_Anchor_List               str
Payment_Timeframe             float64
Registration_Date                 str
Proforma_Account_Stage            str
Proforma_Audit_Status             str
Current_Auto_Renewal_Flag         str
Current_World_Pay_Token           str
Renewal_Scor

Drop columns that are completely null or have unnamed index formats.

In [4]:
# Drop if every value in the column is missing
df_bill = df_bill.dropna(axis=1, how="all")

# Remove Unnamed columns
df_bill = df_bill.loc[:, ~df_bill.columns.str.contains("^Unnamed")]

print(f"Columns remaining after dropping empty/unnamed: {len(df_bill.columns)}")
print(df_bill.columns.tolist())

Columns remaining after dropping empty/unnamed: 58
['Co_Ref', 'Renewal_Month', 'Connection_Net', 'Connection_Qty', 'Discount_Amount', 'Sustainability_Score', 'Total_Renewal_Score_New', 'Starting_Connection_Net', 'Starting_Connection_Qty', 'Last_Years_Price', 'Auto_Renewal_Score', 'Status_Scores', 'Anchoring_Score', 'Tenure_Scores', 'Proforma_Auto_Renewal', 'Proforma_World_Pay_Token', 'Proforma_Date', 'Current_Anchorings', 'Current_Anchor_List', 'Payment_Timeframe', 'Registration_Date', 'Proforma_Account_Stage', 'Proforma_Audit_Status', 'Current_Auto_Renewal_Flag', 'Current_World_Pay_Token', 'Renewal_Score_At_Release', 'Proforma_Membership_Status', 'Proforma_Approved_Lists', 'Tenure_Years', 'Band', 'Prospect_Renewal_Date', 'Closed_Date', 'Prospect_Status', 'Starting_Net', 'Starting_Vat', 'Starting_Gross', 'Starting_Membership_Net', 'Starting_Package_Net', 'Starting_PQQ_Net', 'Gross', 'Membership_Net', 'Package_Net', 'PQQNet', 'Total_Net_Paid', 'Prospect_Outcome', 'Payment_Method', 'Amou

Check for null values and their percentages

In [5]:
null_pct = (df_bill.isnull().sum() / len(df_bill) * 100).round(1)
null_df = pd.DataFrame({"null_count": df_bill.isnull().sum(), "null_pct": null_pct})
print(null_df[null_df["null_count"] > 0].sort_values("null_pct", ascending=False))

                            null_count  null_pct
Connection_Net                  114045      93.4
Connection_Qty                  114045      93.4
Starting_Connection_Net         113537      93.0
Starting_Connection_Qty         113537      93.0
Discount_Amount                 108531      88.9
Last_Renewal                     48291      39.6
Last_Connections                 48380      39.6
Last_Total_Net_Paid              48356      39.6
Last_Band                        48311      39.6
Current_Anchor_List              25957      21.3
Payment_Timeframe                20856      17.1
Total_Net_Paid                   20856      17.1
Proforma_Auto_Renewal            18052      14.8
Proforma_World_Pay_Token         18052      14.8
Proforma_Audit_Status             9229       7.6
Proforma_Account_Stage            9229       7.6
Last_Years_Price                  9090       7.4
Closed_Date                       8188       6.7
Registration_Date                 1018       0.8
Tenure_Group        

Checking unique values for categorical columns to understand the data before cleaning

In [6]:
cat_inspect = [
    "Prospect_Status", "Prospect_Outcome", "Payment_Method", "Band",
    "Current_Auto_Renewal_Flag", "Current_World_Pay_Token",
    "Proforma_Account_Stage", "Proforma_Audit_Status",
    "Proforma_Membership_Status", "Connection_Group", "Tenure_Group",
    "Anchor_Group", "Discount_Amount", "Proforma_Auto_Renewal",
    "Proforma_World_Pay_Token"
]

for col in cat_inspect:
    if col in df_bill.columns:
        print(f"\n--- {col} ---")
        print(df_bill[col].value_counts(dropna=False).head(15))


--- Prospect_Status ---
Prospect_Status
Renewed                           94826
Application and Money In           5897
Renewal Proforma Sent              5219
No Longer Trading                  2387
Do Not Work for Client             2377
Not Value for Money                2196
Refused to Discuss                 1180
Competitor Accreditation            984
Attempted Contact                   925
Intention to Proceed                722
Non Responsive                      698
Not Affordable                      612
Contact Made                        576
Existing Safecontractor Member      436
Cannot Pass Audit                   413
Name: count, dtype: int64

--- Prospect_Outcome ---
Prospect_Outcome
Won        101226
Churned     12668
Open         8188
Name: count, dtype: int64

--- Payment_Method ---
Payment_Method
CARD         71101
BACS         33191
UNKNOWN      16020
WORLD PAY     1699
CHEQUE          71
Name: count, dtype: int64

--- Band ---
Band
Band B     32731
Band C1    249

Dropping columns with over 90% missing values — Connection_Net, Connection_Qty, Starting_Connection_Net,
Starting_Connection_Qty, and Discount_Amount have very little usable data so they will just add noise.

In [7]:
sparse_cols = [
    "Connection_Net", "Connection_Qty",
    "Starting_Connection_Net", "Starting_Connection_Qty",
    "Discount_Amount"
]

df_bill = df_bill.drop(columns=sparse_cols, errors="ignore")
print(f"Shape after dropping sparse cols: {df_bill.shape}")

Shape after dropping sparse cols: (122082, 53)


Parsing date columns into proper datetime objects

In [8]:
date_cols = ["Renewal_Month", "Proforma_Date", "Registration_Date",
             "Prospect_Renewal_Date", "Closed_Date", "DateTime_Out", "Last_Renewal"]

for col in date_cols:
    if col in df_bill.columns:
        df_bill[col] = pd.to_datetime(df_bill[col], dayfirst=True, errors="coerce")

# Quick sanity check on the parsed dates
for col in date_cols:
    if col in df_bill.columns:
        non_null = df_bill[col].notna().sum()
        sample = df_bill[col].dropna().iloc[0] if non_null > 0 else "all null"
        print(f"{col}  | non-null: {non_null}  | sample: {sample}")

Renewal_Month  | non-null: 122082  | sample: 2024-11-01 00:00:00
Proforma_Date  | non-null: 121778  | sample: 2024-09-06 00:00:00
Registration_Date  | non-null: 121064  | sample: 2021-11-05 00:00:00
Prospect_Renewal_Date  | non-null: 122082  | sample: 2024-11-05 00:00:00
Closed_Date  | non-null: 113894  | sample: 2024-11-05 00:00:00
DateTime_Out  | non-null: 122082  | sample: 2024-11-01 00:00:00
Last_Renewal  | non-null: 73791  | sample: 2023-11-01 00:00:00


Standardising boolean-like columns — the raw data uses y/n and True/False inconsistently,
converting them all to clean Yes/No strings for consistency with the other cleaned datasets.

In [9]:
# Current_Auto_Renewal_Flag and Current_World_Pay_Token use y/n
yn_cols = ["Current_Auto_Renewal_Flag", "Current_World_Pay_Token"]
for col in yn_cols:
    df_bill[col] = df_bill[col].map({"y": "Yes", "n": "No"}).fillna("Unknown")

# Proforma_Auto_Renewal and Proforma_World_Pay_Token use True/False
tf_cols = ["Proforma_Auto_Renewal", "Proforma_World_Pay_Token"]
for col in tf_cols:
    df_bill[col] = df_bill[col].astype(str).str.strip()
    df_bill[col] = df_bill[col].map({"True": "Yes", "False": "No"}).fillna("Unknown")

print("Current_Auto_Renewal_Flag:", df_bill["Current_Auto_Renewal_Flag"].value_counts().to_dict())
print("Proforma_Auto_Renewal:", df_bill["Proforma_Auto_Renewal"].value_counts().to_dict())

Current_Auto_Renewal_Flag: {'Yes': 114807, 'No': 7275}
Proforma_Auto_Renewal: {'Yes': 102435, 'Unknown': 18052, 'No': 1595}


Filling null values in categorical columns with sensible defaults

In [10]:
# Columns where "Unknown" makes sense for missing entries
fill_unknown = [
    "Band", "Proforma_Account_Stage", "Proforma_Audit_Status",
    "Proforma_Membership_Status", "Connection_Group",
    "Tenure_Group", "Anchor_Group", "Payment_Method",
    "Current_Anchor_List", "Last_Band"
]

for col in fill_unknown:
    if col in df_bill.columns:
        df_bill[col] = df_bill[col].fillna("Unknown")

# Proforma_Audit_Status has a lowercase "vetting" variant, normalise it
df_bill["Proforma_Audit_Status"] = df_bill["Proforma_Audit_Status"].str.strip()
df_bill.loc[
    df_bill["Proforma_Audit_Status"].str.lower() == "vetting",
    "Proforma_Audit_Status"
] = "Vetting"

print("Nulls in categorical cols after fill:")
for col in fill_unknown:
    if col in df_bill.columns:
        cnt = df_bill[col].isnull().sum()
        print(f"  {col}: {cnt}")

Nulls in categorical cols after fill:
  Band: 0
  Proforma_Account_Stage: 0
  Proforma_Audit_Status: 0
  Proforma_Membership_Status: 0
  Connection_Group: 0
  Tenure_Group: 0
  Anchor_Group: 0
  Payment_Method: 0
  Current_Anchor_List: 0
  Last_Band: 0


Filling missing numeric columns with median values where appropriate,
and 0 where the column represents a count or amount

In [11]:
# Columns where 0 is the natural fill (money not paid, no timeframe, no connections)
fill_zero = ["Total_Net_Paid", "Payment_Timeframe",
             "Last_Total_Net_Paid", "Last_Connections"]
for col in fill_zero:
    if col in df_bill.columns:
        df_bill[col] = df_bill[col].fillna(0)

# Columns where median is a better fallback
fill_median = ["Last_Years_Price", "Renewal_Score_At_Release",
               "Proforma_Approved_Lists", "Tenure_Years", "#_of_Connection"]
for col in fill_median:
    if col in df_bill.columns:
        med = df_bill[col].median()
        df_bill[col] = df_bill[col].fillna(med)
        print(f"{col}: filled with median = {med}")

print("\nRemaining nulls in numeric cols:")
num_cols = df_bill.select_dtypes(include=["number"]).columns
remaining = df_bill[num_cols].isnull().sum()
print(remaining[remaining > 0] if (remaining > 0).any() else "None")

Last_Years_Price: filled with median = 919.0
Renewal_Score_At_Release: filled with median = 26.0
Proforma_Approved_Lists: filled with median = 1.0
Tenure_Years: filled with median = 5.0
#_of_Connection: filled with median = 1.0

Remaining nulls in numeric cols:
None


Converting string columns to category dtype for memory efficiency and downstream modelling

In [12]:
cat_cols = [
    "Prospect_Status", "Prospect_Outcome", "Payment_Method",
    "Band", "Current_Auto_Renewal_Flag", "Current_World_Pay_Token",
    "Proforma_Auto_Renewal", "Proforma_World_Pay_Token",
    "Proforma_Account_Stage", "Proforma_Audit_Status",
    "Proforma_Membership_Status", "Connection_Group",
    "Tenure_Group", "Anchor_Group", "Last_Band"
]

for col in cat_cols:
    if col in df_bill.columns:
        df_bill[col] = df_bill[col].astype("category")

Final verification — checking remaining null values and dtypes after all cleaning

In [13]:
print("Remaining null values:")
print(df_bill.isnull().sum())
print(f"\nTotal rows: {len(df_bill)}")
print(f"Total columns: {len(df_bill.columns)}")

Remaining null values:
Co_Ref                            0
Renewal_Month                     0
Sustainability_Score              0
Total_Renewal_Score_New           0
Last_Years_Price                  0
Auto_Renewal_Score                0
Status_Scores                     0
Anchoring_Score                   0
Tenure_Scores                     0
Proforma_Auto_Renewal             0
Proforma_World_Pay_Token          0
Proforma_Date                   304
Current_Anchorings                0
Current_Anchor_List               0
Payment_Timeframe                 0
Registration_Date              1018
Proforma_Account_Stage            0
Proforma_Audit_Status             0
Current_Auto_Renewal_Flag         0
Current_World_Pay_Token           0
Renewal_Score_At_Release          0
Proforma_Membership_Status        0
Proforma_Approved_Lists           0
Tenure_Years                      0
Band                              0
Prospect_Renewal_Date             0
Closed_Date                    8188
Prosp

In [14]:
df_bill.dtypes

Co_Ref                                   str
Renewal_Month                 datetime64[us]
Sustainability_Score                 float64
Total_Renewal_Score_New              float64
Last_Years_Price                     float64
Auto_Renewal_Score                     int64
Status_Scores                          int64
Anchoring_Score                      float64
Tenure_Scores                        float64
Proforma_Auto_Renewal               category
Proforma_World_Pay_Token            category
Proforma_Date                 datetime64[us]
Current_Anchorings                     int64
Current_Anchor_List                      str
Payment_Timeframe                    float64
Registration_Date             datetime64[us]
Proforma_Account_Stage              category
Proforma_Audit_Status               category
Current_Auto_Renewal_Flag           category
Current_World_Pay_Token             category
Renewal_Score_At_Release             float64
Proforma_Membership_Status          category
Proforma_A

In [15]:
df_bill.info()

<class 'pandas.DataFrame'>
RangeIndex: 122082 entries, 0 to 122081
Data columns (total 53 columns):
 #   Column                      Non-Null Count   Dtype         
---  ------                      --------------   -----         
 0   Co_Ref                      122082 non-null  str           
 1   Renewal_Month               122082 non-null  datetime64[us]
 2   Sustainability_Score        122082 non-null  float64       
 3   Total_Renewal_Score_New     122082 non-null  float64       
 4   Last_Years_Price            122082 non-null  float64       
 5   Auto_Renewal_Score          122082 non-null  int64         
 6   Status_Scores               122082 non-null  int64         
 7   Anchoring_Score             122082 non-null  float64       
 8   Tenure_Scores               122082 non-null  float64       
 9   Proforma_Auto_Renewal       122082 non-null  category      
 10  Proforma_World_Pay_Token    122082 non-null  category      
 11  Proforma_Date               121778 non-null  datet

In [16]:
df_bill.head()

,Co_Ref,Renewal_Month,Sustainability_Score,Total_Renewal_Score_New,Last_Years_Price,Auto_Renewal_Score,Status_Scores,Anchoring_Score,Tenure_Scores,Proforma_Auto_Renewal,...,Connection_Group,Tenure_Group,#_of_Connection,Last_Renewal,Last_Band,Last_Total_Net_Paid,Last_Connections,Anchor_Group,Renewal_Year,DateTime_Out
0,VT6174,2024-11-01,8.0,42.5,799.0,9,9,7.5,9.0,Yes,...,1,3,1.0,2023-11-01,Band B,664.0,1.0,1,2024,2024-11-01
1,VD3828,2025-08-01,8.0,41.5,799.0,9,9,7.5,8.0,Yes,...,1,1,1.0,NaT,Unknown,0.0,0.0,1,2025,2025-08-01
2,DV8120,2025-03-01,8.0,33.0,799.0,8,0,7.5,9.5,Yes,...,1,4+,1.0,2024-03-01,Band C1,749.0,1.0,1,2025,2025-03-01
3,EZ9894,2025-06-01,9.5,44.5,799.0,9,9,7.5,9.5,Yes,...,1,4+,1.0,2024-06-01,Band C1,749.0,1.0,1,2025,2025-06-01
4,FA8957,2025-03-01,9.5,42.5,799.0,9,8,7.5,8.5,Yes,...,1,3,1.0,2024-03-01,Band C1,749.0,1.0,1,2025,2025-03-01


Saving the cleaned dataset

In [17]:
import os

# Ensure the clean data directory exists
clean_data_path = project_root / "data" / "processed"
os.makedirs(clean_data_path, exist_ok=True)

# Write out to clean folder
out_file = clean_data_path / "cleaned_billings.csv"
df_bill.to_csv(out_file, index=False)
print(f"Cleaned dataset saved successfully to {out_file}")

Cleaned dataset saved successfully to c:\Users\MadhavGoyal\Desktop\DS Project'\Churn-Prediction\data\processed\cleaned_billings.csv
